## **Analysis of Acid/Base Catalysis**

The starting point for this script is an .xlsx file with all the experimental parameters as well as the observed rate constants kobs (s-1) from fitting each individual kinetic trace. Based on these numbers, the Jupyter notebook below analyzes the data in five steps. 

**Part 1** imports the relevant data for the analysis. Here, the user specifies which dataset (each in an individual folder) is analyzed.

**Part 2** analyzes all measurements made with the same buffer type AND pH value individually to extract the buffer-independent second-order rate constant k20. As output, it generates a list of all fit results for each pH/pKa combination (buffer_zero_summary.csv) and plots all analyses.

**Part 3** analyzes the data with a pH-rate profile based on the buffer-independent k20 values. It then fits the pH-rate profile to the analytical rate law to extract the rate constants for specific acid catalysis (k3H), specific base catalysis (k3OH), and the water-catalyzed reaction (k2w). It finally plots the pH-rate profile with the fitted line.

**Part 4** extracts the third-order rate constants for general acid and general base catalysis by analyzing each buffer system individually. Here, it imports the rate constants for specific acid catalysis (k3H), specific base catalysis (k3OH), and the water-catalyzed reaction (k2w) that were obtained from the pH-rate profile in Part 3.

**Part 5** analyzes all data in a single step, assuming that acid and base catalysis can be described through a Bronsted correlation. It outputs six parameters that describe the reaction: the slopes/intercept of the two Bronsted correlations, the equilibrium constant, and the rate constant for the uncatalyzed reaction. This is the most complete fit of the data.

All regression analyses in this notebook rely on the Huber robust regression method for solving the equations, typically in a two-step manner: a first crude fit is used to determine the MAE and to flag outliers. In a second step, the data is fitted only with the inliers, with the f_scale factor derived from the MAE. As the data is heteroscedastic, the 95% confidence interval for all fitted parameters is finally determined using a bootstrapping routine.

________



**Part 1.** Define the "dataset", i.e., the folder in which the data is located (e.g., c1). Here, it assumes that the file is named, e.g., c1_data, where c1 would be the name of the dataset. The script then imports the relevant columns from the Excel sheet into a DataFrame. Using the corrected pH and pKₐ values (both adjusted for ionic strength I = 1), it then calculate the proportions of the acidic (HA) and basic (B) forms of the buffer.

In [ ]:
import pandas as pd
from pathlib import Path

# Define the name of the dataset that is analyzed in the whole notebook
dataset = 'c1'  

# 1) Read the raw Excel file, using the first row as the column headers.
file_path = Path(dataset) / f'{dataset}_data.xlsx'
df = pd.read_excel(file_path, header=0)

# 2) Identify which columns should be treated as numeric measurements.
numeric_columns = ['ha_pka', 'pH_corr', 'pka_corr', '[nucleophile]', '[ha]tot', 'k_obs', 'ha_p', 'ha_q']

# 3) Coerce non-numeric entries in those columns to NaN, then drop any incomplete rows.
df_filtered = (
    df[numeric_columns]
    .apply(pd.to_numeric, errors='coerce')
    .dropna()
)

# 4) Preserve the categorical “ha_type” column by aligning on the same indices.
df_filtered['ha_type'] = df.loc[df_filtered.index, 'ha_type']

# 5) Compute the free acid [HA] and base [B] concentrations of the buffer species, where delta = 10**(pKa – pH).
delta = 10 ** (df_filtered['pka_corr'] - df_filtered['pH_corr'])
df_filtered['HA'] = df_filtered['[ha]tot'] * delta / (1 + delta)
df_filtered['B']  = df_filtered['[ha]tot'] - df_filtered['HA']
df_filtered

**Part 2.** For each buffer system at a given pH, first a robust Huber‐loss fit is performed with a large scaling factor is performed to estimate and remove extreme outliers (|residual| > 3×MAE). Then, on the cleaned "inliers", a second fit is performed to determine the second-order rate constant k20 (the hypothetical reaction rate when [HA] = 0), and bootstrap resampling (1000 replicates) is used to compute 95 % confidence intervals. In the plots, excluded points from the first fit are shown in grey, inliers are shown in blue, and a simple regression line is added for visual reference. From this analysis, we only use k20 for the next step - the visualization of the pH-rate profile.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from scipy.stats import linregress
from sklearn.utils import resample
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# -- 1) Model definitions ------------------------------------------------
def p2_calculate_k(params, p2_data):
    """
    Buffer rate model:
      k = (U + 1/eq) * (k3buffer * HA + k20)
    p2_data rows: [pH_corr, U, HA, B]
    """
    _, U, HA, B = p2_data.T
    k3buffer, k20, eq = params
    return (U + 1/eq) * (k3buffer * HA + k20)

def p2_residuals(params, p2_data, p2_k_obs):
    return p2_calculate_k(params, p2_data) - p2_k_obs

def p2_fit_huber(p2_data, p2_k_obs, initial_guess=(1,1,1), f_scale=0.01):
    """
    f_scale is a placeholder - it is calculated for each system in an initial "crude" fit.
    """
    res = least_squares(
        p2_residuals,
        x0=initial_guess,
        args=(p2_data, p2_k_obs),
        loss='huber',
        f_scale=f_scale,
        ftol=1e-10,
        xtol=1e-10,
    )
    return res.x

# -- 2) Prepare container for summary results ----------------------------
p2_summary_rows = []

# -- 3) Loop over each buffer system (pKa, pH) ----------------------------
p2_df_in = df_filtered  # use the filtered DataFrame from Cell 1

for p2_pKa in np.sort(p2_df_in['pka_corr'].unique()):
    for p2_pH in np.sort(p2_df_in.loc[
        p2_df_in['pka_corr'] == p2_pKa, 'pH_corr'
    ].unique()):
        
        # 3a) Subset and assemble data
        p2_subset = p2_df_in[
            (p2_df_in['pka_corr'] == p2_pKa) &
            (p2_df_in['pH_corr']  == p2_pH)
        ]
        p2_data   = p2_subset[['pH_corr','[nucleophile]','HA','B']].values
        p2_k_obs  = p2_subset['k_obs'].values
        p2_label  = p2_subset['ha_type'].iat[0]

        # 3b) Initial Huber(f_scale large) → approx OLS
        p2_params0 = p2_fit_huber(p2_data, p2_k_obs, f_scale=1e6)

        # 3c) Compute MAE & identify inliers
        p2_resid0 = p2_calculate_k(p2_params0, p2_data) - p2_k_obs
        p2_mae    = np.mean(np.abs(p2_resid0))
        if p2_mae == 0:
            p2_mae = np.std(p2_resid0) if np.std(p2_resid0) > 0 else 1e-6

        p2_mask       = np.abs(p2_resid0) <= 3 * p2_mae
        p2_data_in    = p2_data[p2_mask]
        p2_k_inliers  = p2_k_obs[p2_mask]

        # 3d) Final Huber fit on inliers
        p2_popt = p2_fit_huber(
            p2_data_in, p2_k_inliers,
            initial_guess=p2_params0, f_scale=p2_mae
        )

        # 3e) Bootstrap 95% CIs
        p2_n_boot = 1000
        p2_boots  = np.zeros((p2_n_boot, 3))
        for i in range(p2_n_boot):
            idx = resample(np.arange(len(p2_k_inliers)))
            p2_boots[i] = p2_fit_huber(
                p2_data_in[idx],
                p2_k_inliers[idx],
                initial_guess=p2_popt,
                f_scale=p2_mae
            )
        p2_ci_low  = np.percentile(p2_boots,  2.5, axis=0)
        p2_ci_high = np.percentile(p2_boots, 97.5, axis=0)

        # 3f) Record summary
        p2_summary_rows.append({
            'ha_type':       p2_label,
            'pka_corr':      p2_pKa,
            'pH_corr':       p2_pH,
            'k3buffer':      p2_popt[0],
            'k3buffer_lo95': p2_ci_low[0],
            'k3buffer_hi95': p2_ci_high[0],
            'k20':           p2_popt[1],
            'k20_lo95':      p2_ci_low[1],
            'k20_hi95':      p2_ci_high[1],
            'eq':            p2_popt[2],
            'eq_lo95':       p2_ci_low[2],
            'eq_hi95':       p2_ci_high[2],
        })

        # -- 3g) Plot observed vs. predicted ----------------------------
        p2_k_all = p2_calculate_k(p2_popt, p2_data)

        plt.scatter(p2_k_obs[~p2_mask], p2_k_all[~p2_mask],
                    color='lightgrey', zorder=1)
        plt.scatter(p2_k_obs[p2_mask],  p2_k_all[p2_mask],
                    color='b', zorder=2)

        # regression on inliers
        p2_slope, p2_intercept, *_ = linregress(
            p2_k_inliers, p2_k_all[p2_mask]
        )
        x_line = np.linspace(min(p2_k_obs), max(p2_k_obs), 100)
        plt.plot(x_line, p2_slope * x_line + p2_intercept, color='black')

        plt.xlabel('k_observed')
        plt.ylabel('k_predicted')
        plt.tick_params(axis='both', direction='in')

        # diagnostics on inliers
        k_line = p2_slope * p2_k_inliers + p2_intercept
        rss    = np.sum((k_line - p2_k_inliers)**2)
        rmse   = np.sqrt(rss / len(p2_k_inliers))
        r_sq   = 1 - rss / np.sum((p2_k_inliers - p2_k_inliers.mean())**2)

        # stats textbox
        result_text = (
            f"Type of ha: {p2_label}\n"
            f"pKa = {p2_pKa:.2f}, pH = {p2_pH:.2f}\n\n"
            "Optimized Parameters:\n" +
            "".join([
                f"{nm}: {p2_popt[i]:.2e} 95% CI [{p2_ci_low[i]:.2e}, {p2_ci_high[i]:.2e}]\n"
                for i, nm in enumerate(['k3buffer','k20','eq'])
            ]) +
            "\nStatistics based on inliers:\n"
            f"RMSE: {rmse:.2e}\n"
            f"R²:   {r_sq:.3f}\n\n"
            "Legend:\n\n\n\n\n"
        )
        plt.text(1.05, 0.1, result_text,
                 transform=plt.gca().transAxes,
                 fontsize=10, verticalalignment='bottom',
                 bbox=dict(facecolor='white', alpha=0.5))

        # legend
        custom_legend = [
            Line2D([0],[0], marker='o', color='w',
                   label='Outlier (not used for fitting)',
                   markerfacecolor='lightgrey', markersize=8),
            Line2D([0],[0], marker='o', color='w', label='Data',
                   markerfacecolor='blue', markersize=8),
            Line2D([0],[0], color='black', lw=2,
                   label='Linear Regression')
        ]
        plt.legend(handles=custom_legend,
                   bbox_to_anchor=(1.05, 0.1),
                   loc='lower left', frameon=False)
        plt.grid(False)
        plt.show()

# -- 4a) Format and save summary -----------------------------------------
p2_summary_df  = pd.DataFrame(p2_summary_rows)
p2_summary_fmt = p2_summary_df.copy()
for col in p2_summary_fmt.columns:
    if col not in ['ha_type','pka_corr','pH_corr']:
        p2_summary_fmt[col] = p2_summary_fmt[col].map(lambda x: f"{x:.2e}")

p2_summary_fmt.to_excel(f'{dataset}/buffer_zero_summary.xlsx', index=False)
p2_summary_fmt

**Part 3.** In this part, we import the buffer-independent rate constants k<sub>20</sub> calculated in Part 2, and plot their lg-values versus pH to generate a pH–rate profile. A two-step Huber‐loss regression (first with a large scaling factor used to estimate the residual MAE, then with f<sub>scale</sub>=MAE and non-negative constraints) yields fits for k2w, k3h, and k3oh. Then, 1000 bootstrap replicates are performed to derive point-wise 95 % confidence bands around the fitted curve. The final plot shows data points with asymmetric error bars (with the CI from step 2), the Huber fit line, and its confidence ribbon, and the parameters plus CIs are exported for reporting.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from sklearn.utils import resample
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# -- 1) Load the k20 vs pH data and filter out non-positive k20 --------
p3_df_profile = pd.read_excel(f'{dataset}/buffer_zero_summary.xlsx')

# keep *all* rows with k20 > 0, even if their lower‐CI was negative
p3_mask_positive = p3_df_profile['k20'] > 0
p3_df_profile = p3_df_profile.loc[p3_mask_positive].reset_index(drop=True)

p3_pH_vals    = p3_df_profile['pH_corr'].values
p3_k20_obs    = p3_df_profile['k20'].values
p3_logk20_obs = np.log10(p3_k20_obs)

# compute raw CI endpoints
p3_k20_lo95 = p3_df_profile['k20_lo95'].values
p3_k20_hi95 = p3_df_profile['k20_hi95'].values

# build asymmetric error bars on log10 scale:
#   - if lo95>0, compute log10 difference; else set lower‐error to zero
p3_err_low  = np.where(
    p3_k20_lo95 > 0,
    p3_logk20_obs - np.log10(p3_k20_lo95),
    0
)
# upper‐error always computed as k20_hi95 should be > k20_obs
p3_err_high = np.log10(p3_k20_hi95) - p3_logk20_obs

# -- 2) Define the profile model & residuals ----------------------------
def p3_k20_model(p3_params, p3_pH):
    k2w, k3h, k3oh = p3_params
    H   = 10 ** (-p3_pH)
    OH  = 10 ** (-(14 - p3_pH))
    return k2w + k3h * H + k3oh * OH

def p3_resid_profile(p3_params, p3_pH, p3_logk_obs):
    return np.log10(p3_k20_model(p3_params, p3_pH)) - p3_logk_obs

def p3_fit_huber_profile(p3_pH, p3_logk_obs, p3_initial_guess, p3_f_scale):
    """Fit Huber regression with positivity bounds"""
    lb = [0.0, 0.0, 0.0]
    ub = [np.inf, np.inf, np.inf]
    res = least_squares(
        p3_resid_profile,
        x0=p3_initial_guess,
        args=(p3_pH, p3_logk_obs),
        loss='huber',
        f_scale=p3_f_scale,
        bounds=(lb, ub),
        ftol=1e-10,
        xtol=1e-10
    )
    return res.x, res

# -- 3) Initial “OLS‐like” fit (using large f_scale) to estimate a better f_scale via MAE --------
p3_params0, _      = p3_fit_huber_profile(
    p3_pH_vals, p3_logk20_obs,
    p3_initial_guess=(1e-3, 1e4, 1e4),
    p3_f_scale=1e6
)
p3_resid0          = p3_resid_profile(p3_params0, p3_pH_vals, p3_logk20_obs)
p3_mae0            = np.mean(np.abs(p3_resid0))
p3_f_scale_final   = p3_mae0 if p3_mae0 > 0 else 1e-3

# -- 4) Final Huber fit on full dataset -----------------------------
p3_popt, _ = p3_fit_huber_profile(
    p3_pH_vals, p3_logk20_obs,
    p3_initial_guess=p3_params0,
    p3_f_scale=p3_f_scale_final
)

# -- 5) Bootstrap for 95% CIs -----------------------------------------
p3_n_boot = 1000
p3_boots  = np.zeros((p3_n_boot, 3))
for i in range(p3_n_boot):
    p3_idx         = resample(np.arange(len(p3_pH_vals)))
    p3_pH_bs       = p3_pH_vals[p3_idx]
    p3_logk20_bs   = p3_logk20_obs[p3_idx]
    p3_boots[i], _ = p3_fit_huber_profile(
        p3_pH_bs, p3_logk20_bs,
        p3_initial_guess=p3_popt,
        p3_f_scale=p3_f_scale_final
    )
p3_ci_lower = np.percentile(p3_boots,  2.5, axis=0)
p3_ci_upper = np.percentile(p3_boots, 97.5, axis=0)

# -- 6) Build model curve & CI bands ------------------------------------
p3_pH_grid    = np.linspace(p3_pH_vals.min(), p3_pH_vals.max(), 200)
p3_k_grid     = p3_k20_model(p3_popt, p3_pH_grid)
p3_logk_grid = np.log10(p3_k_grid)
p3_boot_preds = np.array([np.log10(p3_k20_model(b, p3_pH_grid)) for b in p3_boots])
p3_lower_band = np.percentile(p3_boot_preds,  2.5, axis=0)
p3_upper_band = np.percentile(p3_boot_preds, 97.5, axis=0)

# -- 7) Plot profile  ---------------------------------------
plt.figure(figsize=(10, 5))

plt.errorbar(
    p3_pH_vals, p3_logk20_obs,
    yerr=[p3_err_low, p3_err_high],
    fmt='o', color='b', capsize=3, zorder=2
)
plt.plot(p3_pH_grid, p3_logk_grid, color='black', zorder=1)
plt.fill_between(
    p3_pH_grid, p3_lower_band, p3_upper_band,
    color='black', alpha=0.2, zorder=0
)

plt.xlabel('pH')
plt.ylabel('log$_{10}$(k$_{20}$)')
plt.tick_params(axis='both', direction='in')

# calculate diagnostics
p3_rss   = np.sum((p3_resid_profile(p3_popt, p3_pH_vals, p3_logk20_obs))**2)
p3_rmse  = np.sqrt(p3_rss / len(p3_pH_vals))
p3_ss_tot= np.sum((p3_logk20_obs - p3_logk20_obs.mean())**2)
p3_r_sq  = 1 - p3_rss/p3_ss_tot
p3_dof   = len(p3_pH_vals) - len(p3_popt)

# output for stats textbox
p3_result_text = f"""
Optimized Parameters:
k2w:  {p3_popt[0]:.2e} 95% CI [{p3_ci_lower[0]:.2e}, {p3_ci_upper[0]:.2e}]
k3H:  {p3_popt[1]:.2e} 95% CI [{p3_ci_lower[1]:.2e}, {p3_ci_upper[1]:.2e}]
k3OH: {p3_popt[2]:.2e} 95% CI [{p3_ci_lower[2]:.2e}, {p3_ci_upper[2]:.2e}]

Statistics:
RSS:  {p3_rss:.2e}
RMSE: {p3_rmse:.2e}
R²:   {p3_r_sq:.3f}
DOF: {p3_dof}

Legend:




"""
_ = plt.text(
    1.05, 0.1, p3_result_text,
    transform=plt.gca().transAxes,
    fontsize=10,
    verticalalignment='bottom',
    bbox=dict(facecolor='white', alpha=0.5)
)

# Legend
p3_custom_legend = [
    Line2D([0],[0], marker='o', color='w', label='Data',
           markerfacecolor='blue', markersize=8),
    Line2D([0],[0], color='black', lw=2, label='Fit line (Huber regression)'),
    Line2D([0],[0], color='black', lw=6, alpha=0.2, label='95% CI')
]
plt.legend(
    handles=p3_custom_legend,
    bbox_to_anchor=(1.05, 0.1),
    loc='lower left',
    borderaxespad=0.,
    frameon=False
)

plt.grid(False)
plt.tight_layout()
plt.savefig(f'{dataset}/ph_rate_profile.png', bbox_inches='tight')
plt.show()

# -- 8) Save fitted parameters and CIs to Excel ----------------------------------
p3_param_dict = {
    'k2w':         [f"{p3_popt[0]:.3e}"],
    'k2w_lo95':    [f"{p3_ci_lower[0]:.3e}"],
    'k2w_hi95':    [f"{p3_ci_upper[0]:.3e}"],
    'k3h':         [f"{p3_popt[1]:.3e}"],
    'k3h_lo95':    [f"{p3_ci_lower[1]:.3e}"],
    'k3h_hi95':    [f"{p3_ci_upper[1]:.3e}"],
    'k3oh':        [f"{p3_popt[2]:.3e}"],
    'k3oh_lo95':   [f"{p3_ci_lower[2]:.3e}"],
    'k3oh_hi95':   [f"{p3_ci_upper[2]:.3e}"],
}
p3_parameters_df = pd.DataFrame(p3_param_dict)
p3_parameters_df.to_excel(f'{dataset}/ph_rate_parameters.xlsx', index=False)

**Part 4.** In this part, the script determines the buffer‐dependent rate constants by fixing the water and hydroxide rate contributions (k₂w, k₃H, k₃OH) to the values determined from the analysis in part 3. For each pKa, only the buffer‐specific rate constants k₃HA and k₃B are fitted (the equilibrium constant eq is also allowed to vary freely) using a two-step Huber regression protocol (all parameters ≥ 0). For this analysis, all rate constants are initially log-scaled. As before, bootstrap resampling is used for each pKa subset (1000 replicates) to obtain 95 % confidence intervals for k₃HA, k₃B, and eq. Lastly, the observed versus predicted k values are plotted for each pKa value, and the results are exported.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from scipy.stats import linregress
from sklearn.utils import resample
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# 0) Load water/hydroxide parameters from pH profile fit (Box 3 output)
p4_water_params = pd.read_excel(f'{dataset}/ph_rate_parameters.xlsx')
p4_k2w_val  = float(p4_water_params.loc[0, 'k2w'])
p4_k3h_val  = float(p4_water_params.loc[0, 'k3h'])
p4_k3oh_val = float(p4_water_params.loc[0, 'k3oh'])

# 1) Define model fixing water terms; fit k3HA, k3B, eq
#    params: [k3HA, k3B, eq]
def p4_calculate_k_buffer_fixed(p4_params, p4_data):
    pH_corr, U, HA, B = p4_data.T
    k3HA, k3B, eq = p4_params
    H  = 10**(-pH_corr)
    OH = 10**(-(14 - pH_corr))
    k20 = p4_k2w_val + p4_k3h_val * H + p4_k3oh_val * OH
    return (U + 1/eq) * (k3HA * HA + k3B * B + k20)

# 2) Residuals for the buffer-fixed model using decadic log scale
def p4_residuals_fixed_log10(p4_params, p4_data, p4_k_obs):
    p4_k_pred = p4_calculate_k_buffer_fixed(p4_params, p4_data)
    # penalize non-positive values
    if np.any(p4_k_pred <= 0) or np.any(p4_k_obs <= 0):
        return np.full_like(p4_k_obs, 1e6)
    return np.log10(p4_k_pred) - np.log10(p4_k_obs)

# 3) Huber‐loss least_squares fit with positivity bounds for fixed buffer model (log10-scale residuals)
def p4_fit_huber_fixed_log10(p4_data, p4_k_obs, p4_x0, p4_f_scale):
    p4_lb = [0]*3
    p4_ub = [np.inf]*3
    p4_res = least_squares(
        p4_residuals_fixed_log10,
        x0=p4_x0,
        args=(p4_data, p4_k_obs),
        loss='huber',
        f_scale=p4_f_scale,
        bounds=(p4_lb, p4_ub),
        ftol=1e-10, xtol=1e-10
    )
    return p4_res.x

# 4) Prepare container for summary results
p4_summary_rows = []
p4_bronsted_rows = []
p4_n_boot = 1000
p4_names = ['k3HA','k3B','eq']

# 5) Loop over each pKa
for p4_pKa_val in np.sort(df_filtered['pka_corr'].unique()):
    # 5a) Subset data for this pKa
    p4_subset = df_filtered[df_filtered['pka_corr'] == p4_pKa_val]
    p4_data   = p4_subset[['pH_corr','[nucleophile]','HA','B']].values
    p4_bronsted_mean = p4_subset[['ha_pka','ha_p','ha_q']].mean().values
    p4_bronsted_rows.append(p4_bronsted_mean)
    #p4_bronsted_data   = p4_subset[['ha_pka','ha_p','ha_q']].values
    p4_k_obs  = p4_subset['k_obs'].values
    p4_ha_type = p4_subset['ha_type'].iat[0]

    # 5b) Initial Huber fit (large f_scale) to estimate residual scale
    p4_x0 = np.ones(3)
    p4_params0 = p4_fit_huber_fixed_log10(p4_data, p4_k_obs, p4_x0, 1e6)
    p4_resid0 = p4_residuals_fixed_log10(p4_params0, p4_data, p4_k_obs)
    p4_mae0   = np.mean(np.abs(p4_resid0))
    p4_f_scale = p4_mae0 if p4_mae0 > 0 else 1e-3

    # 5c) Identify inliers: |resid| ≤ 3×MAE
    p4_mask = np.abs(p4_resid0) <= 3 * p4_mae0
    p4_data_in = p4_data[p4_mask]
    p4_k_in    = p4_k_obs[p4_mask]

    # 5d) Final Huber fit on inliers
    p4_popt = p4_fit_huber_fixed_log10(p4_data_in, p4_k_in, p4_params0, p4_f_scale)

    # 5e) Bootstrap 95% CIs on inliers
    p4_boots = np.zeros((p4_n_boot, 3))
    for i in range(p4_n_boot):
        p4_idx   = resample(np.arange(len(p4_k_in)))
        p4_boots[i] = p4_fit_huber_fixed_log10(
            p4_data_in[p4_idx], p4_k_in[p4_idx],
            p4_popt, p4_f_scale
        )
    p4_ci_low  = np.percentile(p4_boots,  2.5, axis=0)
    p4_ci_high = np.percentile(p4_boots, 97.5, axis=0)

    # 5f) Record summary
    p4_summary_rows.append({
        'ha_type':       p4_ha_type,
        'pka_corr':      p4_pKa_val,
        **{nm: p4_popt[i]             for i,nm in enumerate(p4_names)},
        **{f"{nm}_lo95": p4_ci_low[i]  for i,nm in enumerate(p4_names)},
        **{f"{nm}_hi95": p4_ci_high[i] for i,nm in enumerate(p4_names)},
    })

    # 5g) Plot log10–log10 observed vs predicted
    p4_k_all = p4_calculate_k_buffer_fixed(p4_popt, p4_data)
    p4_log_obs = np.log10(p4_k_obs)
    p4_log_pred = np.log10(p4_k_all)

    plt.figure()
    plt.scatter(p4_log_obs[~p4_mask], p4_log_pred[~p4_mask], color='lightgrey', zorder=1, label='Outliers')
    plt.scatter(p4_log_obs[p4_mask],  p4_log_pred[p4_mask],  color='b',      zorder=2, label='Inliers')

    # regression on log10-scale inliers
    p4_slope, p4_intercept, p4_r_value, p4_p_value, p4_stderr = linregress(
        p4_log_obs[p4_mask], p4_log_pred[p4_mask]
    )
    p4_x_line = np.linspace(p4_log_obs.min(), p4_log_obs.max(), 100)
    p4_y_line = p4_slope * p4_x_line + p4_intercept
    plt.plot(p4_x_line, p4_y_line, color='black')

    plt.xlabel('log$_{10}$(k_observed)')
    plt.ylabel('log$_{10}$(k_predicted)')
    plt.tick_params(axis='both', direction='in')

    # 5h) Compute diagnostics on inliers
    p4_log_obs_in  = p4_log_obs[p4_mask]
    p4_log_pred_in = p4_log_pred[p4_mask]

    # Huber‐fit diagnostics in log10-space
    p4_rss_log  = np.sum((p4_log_pred_in - p4_log_obs_in)**2)
    p4_tss_log  = np.sum((p4_log_obs_in - p4_log_obs_in.mean())**2)
    p4_rmse_log = np.sqrt(p4_rss_log / len(p4_log_obs_in))
    p4_r2_log   = 1 - p4_rss_log / p4_tss_log

    # OLS regression diagnostics
    p4_r2_lr = p4_r_value**2

    # 5i) Stats textbox (updated reporting)
    p4_text = (
        f"""
Type of ha: {p4_ha_type}
pKa = {p4_pKa_val:.2f}

Optimized Parameters:
""" +
        "".join([f"{nm}: {p4_popt[i]:.2e} 95% CI [{p4_ci_low[i]:.2e}, {p4_ci_high[i]:.2e}]\n"
                  for i,nm in enumerate(p4_names)]) +
        f"""
Huber‐fit diagnostics (log10-space):
  RMSE = {p4_rmse_log:.2e}, R² = {p4_r2_log:.3f}

Regression line of lg(k_predicted) vs lg(k_observed):
  slope = {p4_slope:.3f}, R² = {p4_r2_lr:.3f}

Legend:
\n\n\n\n\n
"""
    )
    plt.text(1.05, 0.5, p4_text,
             transform=plt.gca().transAxes,
             fontsize=10,
             verticalalignment='center',
             bbox=dict(facecolor='white', alpha=0.5))

    # 5j) Custom legend
    p4_custom_legend = [
        Line2D([0],[0], marker='o', color='w', label='Outliers',      markerfacecolor='lightgrey', markersize=8),
        Line2D([0],[0], marker='o', color='w', label='Inliers',       markerfacecolor='blue',       markersize=8),
        Line2D([0],[0], color='black',      lw=2,  label='Linear fit line')
    ]
    plt.legend(handles=p4_custom_legend,
               bbox_to_anchor=(1.05, 0.1), loc='lower left', frameon=False)
    plt.grid(False)

    plt.savefig(f'{dataset}/fit_buffer_pKa_{p4_pKa_val:.2f}.png', bbox_inches='tight')
    plt.show()

# 6a) Format all numeric columns in 2-decimal scientific notation
p4_summary_df2   = pd.DataFrame(p4_summary_rows)
p4_bronsted_data = np.array(p4_bronsted_rows)
p4_bronsted_df = pd.DataFrame(p4_bronsted_data, columns=['ha_pka', 'ha_p', 'ha_q'])
p4_summary_df3 = pd.concat([p4_summary_df2, p4_bronsted_df], axis=1)

# === 6) Add water/oxonium/hydroxide parameters to p4_summary_df3 ===

p4_water_params = pd.read_excel(f'{dataset}/ph_rate_parameters.xlsx')
water_dict = p4_water_params.iloc[0].to_dict()

# Create water-type entries manually, mapped to correct fields
rows_water_extra = []

# 1. Water row
rows_water_extra.append({
    'ha_type': 'water',
    'pka_corr': 7.00,
    'ha_pka': 7.00,
    'ha_p': 0,
    'ha_q': 0,
    'k3HA': water_dict.get('k2w', 0),
    'k3HA_lo95': water_dict.get('k2w_lo95', 0),
    'k3HA_hi95': water_dict.get('k2w_hi95', 0),
    'k3B': 0,
    'k3B_lo95': 0,
    'k3B_hi95': 0,
    'eq': 0,
    'eq_lo95': 0,
    'eq_hi95': 0,
})

# 2. Oxoniumion row
rows_water_extra.append({
    'ha_type': 'oxoniumion',
    'pka_corr': -1.74,
    'ha_pka': -1.74,
    'ha_p': 1,
    'ha_q': 1,
    'k3HA': water_dict.get('k3h', 0),
    'k3HA_lo95': water_dict.get('k3h_lo95', 0),
    'k3HA_hi95': water_dict.get('k3h_hi95', 0),
    'k3B': 0,
    'k3B_lo95': 0,
    'k3B_hi95': 0,
    'eq': 0,
    'eq_lo95': 0,
    'eq_hi95': 0,
})

# 3. Hydroxide row
rows_water_extra.append({
    'ha_type': 'hydroxide',
    'pka_corr': 15.74,
    'ha_pka': 15.74,
    'ha_p': 1,
    'ha_q': 1,
    'k3HA': 0,
    'k3HA_lo95': 0,
    'k3HA_hi95': 0,
    'k3B': water_dict.get('k3oh', 0),
    'k3B_lo95': water_dict.get('k3oh_lo95', 0),
    'k3B_hi95': water_dict.get('k3oh_hi95', 0),
    'eq': 0,
    'eq_lo95': 0,
    'eq_hi95': 0,
})

# Convert and append to existing DataFrame
water_extra_df = pd.DataFrame(rows_water_extra)
p4_summary_df3 = pd.concat([p4_summary_df3, water_extra_df], ignore_index=True)


p4_summary_fmt2  = p4_summary_df3.copy()
for p4_col in p4_summary_fmt2.columns:
    if p4_col not in ['ha_type', 'pka_corr']:
        p4_summary_fmt2[p4_col] = p4_summary_fmt2[p4_col].map(lambda x: f"{x:.2e}")

# 6b) Write the formatted frame to Excel
p4_summary_fmt2.to_excel(f'{dataset}/general_catalysis_analysis.xlsx', index=False)
p4_summary_fmt2

**Part 5.** This final part analyzes the whole dataset at once (see the Supporting Information of the paper for a description), assuming that both acid and base catalysis can be described by a Brönsted correlation. In this way, the observed rate constants are modeled with six parameters: two Brønsted slopes (α₁, α₂) and intercepts (C₁, C₂), one effective global equilibrium constant eq, and the uncatalyzed rate k₂w. A two-step Huber regression protocol (first estimating the MAE and flagging outliers with |Δlog₁₀k| > 3 MAE, then refitting on inliers) and bootstrapping (1000 replicates) are used to obtain 95 % CIs for each parameter. 

**Note:** Brønsted plots are constructed with the pKa values at ionic strength = 0. Accordingly, ha_pka is now used for the plot (but HA and B are still calculated from the actual pKa value of the measurement).

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from scipy.stats import linregress
from sklearn.utils import resample
import matplotlib.pyplot as plt

# Define the function to calculate k from the model. This now uses the uncorrected pKa at I = 0
def p5_calculate_k(params, data):
    pH, Nu, HA, B, ha_pka, p, q = data.T
    k2w, alpha1, alpha2, C1, C2, eq = params 
    return (Nu + 1/eq) * (
        (10**(alpha1*(ha_pka + np.log10(p/q)) + C1) * HA) +
        (10**(alpha2*(ha_pka + np.log10(p/q)) + C2) * B) +
        k2w +
        ((10**(alpha1*(-1.74) + C1)) * 10**(-pH) +
         (10**(alpha2*(15.74) + C2)) * 10**(-(14 - pH)))
    )

# Residuals in log10-space for Huber loss
def p5_residuals_log10(params, data, k_obs):
    k_pred = p5_calculate_k(params, data)
    if np.any(k_pred <= 0) or np.any(k_obs <= 0):
        return np.full_like(k_obs, 1e6)
    return np.log10(k_pred) - np.log10(k_obs)

# Huber‐loss wrapper
def p5_fit_huber(data, k_obs, x0, f_scale):
    lb = [0, -1, 0, -np.inf, -np.inf, 0]
    ub = [np.inf,  0, 1,  np.inf,  np.inf, np.inf]
    return least_squares(
        p5_residuals_log10, x0=x0,
        args=(data, k_obs),
        loss='huber', f_scale=f_scale,
        bounds=(lb, ub),
        ftol=1e-12, xtol=1e-12, gtol=1e-12
    )

# --- use df_filtered from above ---
p5_data = df_filtered[['pH_corr','[nucleophile]','HA','B','ha_pka','ha_p','ha_q']].values
p5_k_obs = df_filtered['k_obs'].values

# 1) initial crude Huber fit that behaves OLS-like
p5_res0   = p5_fit_huber(p5_data, p5_k_obs, x0=(0.01,-0.5,0.5,0,1,1), f_scale=1e6)
p5_resid0 = p5_residuals_log10(p5_res0.x, p5_data, p5_k_obs)
p5_mae0   = np.mean(np.abs(p5_resid0))
p5_f_scale= p5_mae0 if p5_mae0>0 else 1e-3

# 2) identify inliers
p5_mask   = np.abs(p5_resid0) <= 3*p5_mae0
p5_data_in, p5_k_in = p5_data[p5_mask], p5_k_obs[p5_mask]

# 3) final Huber fit on inliers
p5_res1 = p5_fit_huber(p5_data_in, p5_k_in, x0=p5_res0.x, f_scale=p5_f_scale)
p5_popt = p5_res1.x

# 4) bootstrap 95% CIs
p5_n_boot = 1000
p5_boots  = np.zeros((p5_n_boot, len(p5_popt)))
for i in range(p5_n_boot):
    idx         = resample(np.arange(len(p5_k_in)))
    p5_boots[i] = p5_fit_huber(p5_data_in[idx], p5_k_in[idx], x0=p5_popt, f_scale=p5_f_scale).x
p5_ci_low, p5_ci_high = np.percentile(p5_boots, [2.5,97.5], axis=0)

# 5) diagnostics
p5_resid_full = p5_residuals_log10(p5_popt, p5_data, p5_k_obs)
p5_rss   = np.sum(p5_resid_full**2)
p5_rmse  = np.sqrt(p5_rss / len(p5_k_obs))
p5_tss   = np.sum((np.log10(p5_k_obs) - np.log10(p5_k_obs).mean())**2)
p5_r2    = 1 - p5_rss/p5_tss
p5_N_in  = p5_mask.sum(); p5_N_tot = len(p5_k_obs)
p5_mae_in= np.mean(np.abs(p5_resid0[p5_mask]))

# 6) prepare plotting arrays
p5_log_obs  = np.log10(p5_k_obs)
p5_log_pred = np.log10(p5_calculate_k(p5_popt, p5_data))

# 7) regression on inliers
p5_slope, p5_intercept, p5_r_val, _, _ = linregress(p5_log_obs[p5_mask], p5_log_pred[p5_mask])

# 8) plotting with constrained_layout to reserve room for stats box
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)

# scatter outliers/inliers
ax.scatter(p5_log_obs[~p5_mask], p5_log_pred[~p5_mask],
           color='lightgrey', label='Outliers')
ax.scatter(p5_log_obs[p5_mask],  p5_log_pred[p5_mask],
           color='blue',      label='Inliers')

# regression line
x_vals = np.linspace(p5_log_obs.min(), p5_log_obs.max(), 100)
ax.plot(x_vals, p5_slope*x_vals + p5_intercept,
        color='black', label='Linear fit')

# labels & legend
ax.set_xlabel('log$_{10}$(k_observed)')
ax.set_ylabel('log$_{10}$(k_predicted)')
ax.legend(loc='upper left')
ax.grid(False)

# 9) stats textbox
p5_stats_text = (
    "Optimized Parameters:\n"
    + "".join(
        f"{nm}: {p5_popt[i]:.2e} 95% CI [{p5_ci_low[i]:.2e}, {p5_ci_high[i]:.2e}]\n"
        for i, nm in enumerate(['k2w','α₁','α₂','C₁','C₂','eq'])
    )
    + "\n"
    + f"Inliers/Total: {p5_N_in}/{p5_N_tot}\n\n"
    + "Huber fit diagnostics (log10-space):\n"
    + f"  MAE = {p5_mae_in:.2e}\n"
    + f"  RMSE= {p5_rmse:.2e}\n"
    + f"  R²  = {p5_r2:.3f}\n\n"
    + "Regression line of lg(k_predicted) vs lg(k_observed):\n"
    + f"  slope = {p5_slope:.3f}, R² = {p5_r_val**2:.3f}"
)

ax.text(
    1.02, 0.5, p5_stats_text,
    transform=ax.transAxes,
    fontsize=10, va='center',
    bbox=dict(facecolor='white', alpha=0.5)
)

# save with the original aspect‐ratio, no clipping
fig.savefig(f'{dataset}/fit_broensted_model.png', dpi=200,bbox_inches='tight')
plt.show()